# LLM Benchmarking: Neural Metrics and Analysis Notebook

Runs the **GPU-only** part of the LLM benchmark (COMET, BERTScore, CometKiwi) in the
same environment as the NMT study (`requirements-metrics.txt`), then regenerates every
table and figure. The lexical metrics already run locally on CPU (see `README.md`).

Replace `YOUR_HF_TOKEN` with your own token and never commit it. Request access to the gated
models `Unbabel/wmt22-comet-da` and `Unbabel/wmt22-cometkiwi-da` first.

**Layout expected:** this repo (`llm models/`) and the NMT repo (`make it like/`) side by side,
or set `NMT_REPO` to the NMT repo path. The NMT repo is only read, never modified.

## 0. Get both repositories
Upload or clone both folders into `/content`.

In [ ]:
%cd /content
# !git clone <this LLM repo> "llm models"
# !git clone <NMT repo> "make it like"
import os
os.environ["NMT_REPO"] = "/content/make it like"
%cd "/content/llm models"

## 1. Metrics environment (GPU)
Same pins as the NMT metric run. `setuptools<81` keeps `pkg_resources` for the COMET import chain.

In [ ]:
!pip install -q -r requirements-metrics.txt
import torch; print("CUDA:", torch.cuda.is_available())
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

## 2. Neural and reference-free metrics
The lexical metrics are already in `metrics/results/metric_scores_lexical.csv`; recompute them here only if you want a single-environment record.

In [ ]:
# !python scripts/compute_metrics.py --all --metrics lexical
!python scripts/compute_metrics.py --all --metrics neural     # COMET + BERTScore
!python scripts/compute_cometkiwi.py --all                    # resumable, incremental save

## 3. Merge, significance, tables and figures

In [ ]:
!python scripts/merge_metric_results.py
!python scripts/make_pivots.py
%cd scripts
!python significance.py
!python analysis.py
!python error_analysis.py
%cd ..
!pip freeze > requirements-metrics-llm-run.txt

## Adding another LLM (e.g. Llama 3.2, TranslateGemma)
1. Put its `translations_<language>.csv` files (test_run.py format) in a folder, e.g. `Llama3.2/`.
2. `python scripts/prepare_llm_outputs.py --raw-dir Llama3.2 --system llama32_3b --clean-variant`
3. If `MISSING.csv` is non-empty: `python scripts/fill_missing_llm.py --system llama32_3b --model llama3.2`
4. Add the system to `LLM_SYSTEMS` in `scripts/common.py`, `SYSTEMS` in `compute_metrics.py` / `compute_cometkiwi.py`, plus a display name and colour in `common.py`.
5. Re-run sections 2-3.